# Import

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
%matplotlib inline 
# pour afficher facilement les graphiques 
from urllib.request import urlopen
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
import os
import re
from pathlib import Path
from nltk.stem import SnowballStemmer
import spacy


# Variables 

In [38]:
stemmer = SnowballStemmer("french")
nlp = spacy.load("fr_core_news_sm")

In [39]:
try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    BASE_DIR = Path.cwd().parent

BULLETINS = BASE_DIR / "BULLETINS"
DATA = BASE_DIR / "data"
OUTPUT = BASE_DIR / "output"

# 0. Découpage du corpus 

comment est ce qu'on vas gérer les images ? 
est ce que c'est pertinent de melanger 

In [40]:
# permet d'indenter le xml 
def indent(elem, level=0):
    i = "\n" + level*"  "
    if len(elem):
        if not elem.text or not elem.text.strip():
            elem.text = i + "  "
        for e in elem:
            indent(e, level+1)
        if not e.tail or not e.tail.strip():
            e.tail = i
    if level and (not elem.tail or not elem.tail.strip()):
        elem.tail = i

def decoupage(corpus, balise):
    try : 
        # obtenir le code html de la page
        with open(corpus, "r", encoding = "UTF8") as f :
            html = f.read()

        # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
        soup = BeautifulSoup(html, 'html.parser' )
        type(soup)
        root = ET.Element("corpus")
        corpus = soup.corpus
        all_documents = corpus.find_all("document")
        for doc in all_documents :
            num_article = doc.article.get_text()
            # val_balise = doc.balise.get_text()    # ici comment faire pour qu'il prenne le pbalise du paramettre ?
            val_balise = doc.find(balise).get_text()
            document = ET.SubElement(root, "document")
            article = ET.SubElement(document, "article")
            article.text = num_article
            bal = ET.SubElement(document,balise)
            bal.text = val_balise
        # Écrire dans un fichier
        indent (root)
        tree = ET.ElementTree(root)
        tree.write(OUTPUT/f"{balise}.xml", encoding="utf-8", xml_declaration=True)
            

    except TypeError as e :
        print("erreur : ", e)

corpus = OUTPUT /"corpus.xml"
decoupage(corpus, "titre")


In [41]:
from nltk.stem import SnowballStemmer

stemmer = SnowballStemmer("french")

mots = ["manger", "mangeons", "mangé", "mangeait"]

for mot in mots:
    print(mot, "→", stemmer.stem(mot))

texte = "Le chat mange une pomme dans le jardin"

doc = nlp(texte)

mots_utiles = []

for token in doc:
    if not token.is_stop and token.is_alpha:
        mots_utiles.append(token.lemma_)

print(mots_utiles)

manger → mang
mangeons → mangeon
mangé → mang
mangeait → mang
['chat', 'manger', 'pomme', 'jardin']


# 1. extraction

In [42]:
# def export(file_name, nb_colonnes, liste_finale):
#     if nb_colonnes > 1 :
#         with open(file_name, "w") as f:
#             for element in liste_finale :
#                 line = ""
#                 for i in range(nb_colonnes) :
#                     line = line + f"{element[i]} \t" 
#                 line = line + " \n"
#                 f.writelines(line)
#     if nb_colonnes == 1 :
#         with open(file_name, "w") as f:
#             for element in liste_finale :
#                 line = f"{element} \t" 
#                 f.writelines(line)

def export(file_name, nb_colonnes, liste_finale):
    if nb_colonnes > 1:
        with open(file_name, "w", encoding="utf-8") as f:
            for element in liste_finale:
                line = ""
                for i in range(nb_colonnes):
                    line += f"{element[i]}\t"
                line += "\n"
                f.write(line)

    elif nb_colonnes == 1:
        with open(file_name, "w", encoding="utf-8") as f:
            for element in liste_finale:
                line = f"{element}\n"
                f.write(line)
                
def extraction_spacy(fichier):
    try : 
        # obtenir le code html de la page
        with open(fichier, "r", encoding = "UTF8") as f :
            html = f.read()

        # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
        soup = BeautifulSoup(html, 'html.parser' )
        type(soup)

        #recuperaion du texte 

        liste_finale = []
        all_document = soup.find_all("document")
        print(all_document)
        for document in all_document : 
            print(document)
            # recuperation de l'id article
            article = document.article.get_text()

            # recuperaiton du titre
            titre = document.titre.get_text()
            titre = titre.split()

            #recupertation du texte 
            texte = document.texte.get_text()
            texte = titre.split()

            #recupertation du texte 
            texte = document.rubrique.get_text()
            texte = titre.split()

            texte_document = texte + titre
            doc = nlp(texte_document)

            for token in doc :
                cle = [ article, token.text, token.lemma_]
                liste_finale.append(cle)
        
        export(DATA/f"ExtracionSpacy.txt", 3, liste_finale)

    except Exception as e:

        print("erreur  : ", e)
        print(fichier)

def extraction_snowball(fichier):
    try : 
        # obtenir le code html de la page
        with open(fichier, "r", encoding = "UTF8") as f :
            html = f.read()

        # cree un objet beautifulSoup en transmettant le code html à la fonction BeautifulSoup()
        soup = BeautifulSoup(html, 'html.parser' )
        type(soup)

        # recuperaiton du titre
        titre = soup.title.get_text()
        titre = titre.split()

        #recuperaion du texte 

        liste_finale = []
        all_document = soup.find_all("document")
        for document in all_document : 
            # recuperation de l'id article
            article = document.article.get_text()

            # recuperaiton du titre
            titre = document.title.get_text()
            titre = titre.split()

            #recupertation du texte 
            texte = document.title.get_text()
            texte = titre.split()

            texte_document = texte + titre

            for mot in texte_document :
                if mot.isalpha():  # verrifie si le mot n'est constituer que des lettre et pas des chiffres
                    cle = [article , mot, stemmer.stem(mot.lower())] # mot et sa racine
                    liste_finale.append(cle)
        
        export(DATA/"ExtracionSpacy.txt", 3, liste_finale)

    except Exception as e:

        print("erreur  : ", e)
        print(fichier)


In [43]:
def extraction_spacy(fichier):
    try:
        # Lecture du fichier
        try:
            with open(fichier, "r", encoding="UTF-8") as f:
                html = f.read()
        except FileNotFoundError:
            print(f"Fichier introuvable : {fichier}")
            return
        except Exception as e:
            print(f"Erreur lors de la lecture du fichier : {e}")
            return

        # Parsing HTML/XML
        try:
            soup = BeautifulSoup(html, 'html.parser')
        except Exception as e:
            print(f"Erreur lors du parsing avec BeautifulSoup : {e}")
            return

        liste_finale = []
        all_document = soup.find_all("document")

        if not all_document:
            print("Aucun document trouvé")
            return

        for document in all_document:
            try:
                # Article
                article_tag = document.find("article")
                article = article_tag.get_text() if article_tag else "UNKNOWN"

                # Titre
                titre_tag = document.find("titre")
                titre = titre_tag.get_text().split() if titre_tag else []

                # Texte
                texte_tag = document.find("texte")
                texte = texte_tag.get_text().split() if texte_tag else []

                # Rubrique
                rubrique_tag = document.find("rubrique")
                rubrique = rubrique_tag.get_text().split() if rubrique_tag else []

                # Fusion des contenus
                texte_document = " ".join(texte + titre + rubrique)

                # Traitement spaCy
                try:
                    doc = nlp(texte_document)
                except Exception as e:
                    print(f"Erreur spaCy sur article {article} : {e}")
                    continue

                for token in doc:
                    cle = [article, token.lemma_]
                    liste_finale.append(cle)

            except Exception as e:
                print(f"Erreur sur un document : {e}")
                continue

        # Export
        try:
            
            export(DATA / "lemme.txt", 2, liste_finale)
        except Exception as e:
            print(f"Erreur lors de l'export : {e}")

    except Exception as e:
        print("Erreur globale :", e)
        print(fichier)

In [44]:
extraction_spacy(OUTPUT/"corpus_filtre.xml")

# partie analyse 

** Nombre de mots uniques

    mesure la taille du vocabulaire

    élevé → vocabulaire riche mais dispersé

    faible → mots bien regroupés mais risque de perte d’infocritère 

Fréquence des mots (distribution)

    mesure combien de fois les mots apparaissent

    bonne distribution → mots bien regroupés

    mauvaise distribution → mots éclatés ou mal normalisés

Taux de réduction du vocabulaire

    mesure combien de mots ont été fusionnés

    élevé → compression forte mais perte de sens possible

    modéré → bon équilibre

In [45]:
def analyse(fichier, methode):
    with open(fichier) as f :
        lines = f.readlines()
    
    # je sépare le fichier : je met les mots brute dans une 
    # liste et les reduction dans une autres pour faciliter les traitements
    mots = []
    reductions = []
    for line in lines :
        mots.append(line.split()[0])
        reductions.append(line.split()[1])

    # nombre de mot = nombre de mot unique pour les reduction 
    # car il y a potentiellement les doublons 
    nb_mots = len(reductions)
    nb_unique = len(set(reductions))
    print( f"nombre mots pour la methode {methode} : ",nb_mots )
    print( f"nombre mots unique pour la methode {methode} : ",nb_unique )

    # taux  de perte 
    taux = (1 - (nb_unique/nb_mots))*100
    print( f"taux de perte pour la methode {methode} : ", round(taux,2) ) # roud == deux chiffres apres la virgule

    # dsitribution : frequence d'apparition
    dictionnaire = {}
    for mot in reductions :
        if mot not in dictionnaire :
            dictionnaire[mot] = reductions.count(mot)

    print("distribution : -------------")
    for mot, freq in dictionnaire.items():
        print(mot, ":", freq  )
    print("distribution : -------------")

